# 04 · 토론 파이프라인

**`DebatePipeline`** 클래스를 사용합니다.

| 모드 | 흐름 |
|---|---|
| AI vs AI | 입장 제시 → 찬성 라운드(3턴) → 반대 라운드(3턴) → 주장 다지기 |
| AI vs User | 입장 제시 → 사용자 라운드 → AI 라운드 → 피드백 |

**RDB**: DEBATES / DEBATE_PARTICIPANTS / DEBATE_MESSAGES 저장

## 0. 셋업

In [3]:
import sys, json, textwrap
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from openai import OpenAI
from config import OPENAI_API_KEY, DB_URL
from db.rdb import get_engine, init_tables, load_cards, load_card_tabs
from db.vectordb import get_chroma_client
from pipelines.debate import DebatePipeline

client = OpenAI(api_key=OPENAI_API_KEY)
engine = get_engine(DB_URL)
chroma = get_chroma_client()
init_tables(engine)

best_file = Path("./output/best_combo.txt")
combo     = best_file.read_text().strip() if best_file.exists() else "sentence_large"
parts     = combo.rsplit("_", 1)
STRATEGY, MODEL_KEY = parts[0], parts[1]

debate_pipeline = DebatePipeline(
    engine=engine, chroma=chroma, openai_client=client,
    strategy=STRATEGY, model_key=MODEL_KEY,
)
print(f"DebatePipeline 초기화 완료  (strategy={STRATEGY}, model={MODEL_KEY})")

# ── 토론 출력 헬퍼 ──────────────────────────────────────────────────────────
TYPE_LABEL = {
    "CLAIM":    "주장",
    "REBUTTAL": "반박",
    "ANSWER":   "답변",
    "position": "입장",
    "argument": "주장",
    "rebuttal": "반박",
    "response": "답변",
    "extra_reb": "추가반박",
    "extra_res": "추가답변",
    "question_ans": "질문답변",
}

def print_debate_history(history, title="토론 전체 이력"):
    """토론 이력을 보기 좋게 출력"""
    SEP  = "─" * 64
    SEP2 = "═" * 64
    print(f"\n{SEP2}")
    print(f"  {title}  ({len(history)}개 발언)")
    print(SEP2)
    for i, msg in enumerate(history, 1):
        is_pro  = msg.get("stance", "") == "AGREE"
        side    = "✅ 찬성" if is_pro else "❌ 반대"
        role    = msg.get("role_name", "")
        mtype   = TYPE_LABEL.get(msg.get("message_type", ""), msg.get("message_type", ""))
        p_type  = "👤" if msg.get("participant_type") == "USER" else "🤖"
        content = msg.get("content", "")
        print(f"\n[{i:02d}] {side}  {p_type} {role}  ·  {mtype}")
        print(SEP)
        # 긴 문장 자동 줄바꿈
        for line in textwrap.wrap(content, width=80):
            print(f"  {line}")
    print(f"\n{SEP2}\n")

def print_summary(summary, mode="AI vs AI"):
    """주장 다지기 출력"""
    SEP2 = "═" * 64
    print(f"\n{SEP2}")
    print(f"  ━━ 주장 다지기 ({mode}) ━━")
    print(SEP2)
    print(f"\n[전체 요약]")
    for line in textwrap.wrap(summary.get("overview", ""), width=76):
        print(f"  {line}")

    print("\n[찬성 측 핵심 주장]")
    for a in summary.get("pro_summary", {}).get("key_arguments", []):
        for j, line in enumerate(textwrap.wrap(a, width=74)):
            print(f"  {'•' if j==0 else ' '} {line}")

    print("\n[반대 측 핵심 주장]")
    for a in summary.get("con_summary", {}).get("key_arguments", []):
        for j, line in enumerate(textwrap.wrap(a, width=74)):
            print(f"  {'•' if j==0 else ' '} {line}")

    if mode != "AI vs AI":
        feedback = summary.get("user_feedback", {})
        print("\n[사용자 피드백]")
        print("  잘한 점:")
        for p in feedback.get("strong_points", []):
            print(f"    ✔ {p}")
        print("  보완할 점:")
        for p in feedback.get("weak_points", []):
            print(f"    △ {p}")
    print(f"\n{SEP2}\n")


IndentationError: unindent does not match any outer indentation level (rdb.py, line 529)

## 1. 토론 대상 정책 카드 로드

In [ ]:
# RDB에서 카드 로드 (없으면 mock 사용)
db_cards = load_cards(engine, status="DRAFT", limit=5)

if db_cards:
    card_row  = db_cards[0]
    card_tabs = load_card_tabs(engine, card_row["id"])
    summary_raw = card_tabs.get("SUMMARY", "{}")
    try:
        summary = json.loads(summary_raw)
    except:
        summary = {"title": summary_raw[:50]}

    POLICY_CARD = {
        "id":       card_row["id"],
        "title":    summary.get("title", ""),
        "summary_points": summary.get("summary_points", []),
        "background": card_tabs.get("CORE", ""),
    }
else:
    # Mock 카드 (DB에 카드 없을 때)
    POLICY_CARD = {
        "id": 0,
        "title": "청년 월세 보조금 확대 법안",
        "summary_points": ["월 20만원 보조", "소득 기준 중위 150% 이하", "2년 지원"],
        "background": "청년 주거비 부담 완화를 위해 월세 보조금을 현행 10만원에서 20만원으로 확대하는 법안입니다.",
    }

print(f"토론 주제: {POLICY_CARD['title']}")
print(f"요약: {POLICY_CARD.get('summary_points', [])}")


토론 주제: 청년 월세 보조금 확대 법안
요약: ['월 20만원 보조', '소득 기준 중위 150% 이하', '2년 지원']


## 2. AI vs AI 토론

In [ ]:
# 토론방 생성
debate_id, pro_id, con_id = debate_pipeline.create_ai_vs_ai(
    selected_policy=POLICY_CARD["title"],
)
print(f"토론방 생성: #{debate_id}  (찬성 참여자 #{pro_id}, 반대 #{con_id})")


토론방 생성: #12  (찬성 참여자 #23, 반대 #24)


In [ ]:
# 전체 토론 실행 (약 2~4분 소요, API 비용 발생)
print("AI vs AI 토론 실행 중...")
result = debate_pipeline.run_ai_vs_ai_full(
    debate_id=debate_id,
    policy_card=POLICY_CARD,
    pro_id=pro_id,
    con_id=con_id,
    pro_extra=0,   # 추가 라운드 (0~2)
    con_extra=0,
)

Path("./output/debate_ai_vs_ai.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2)
)
print("토론 결과 저장 완료")


AI vs AI 토론 실행 중...


토론 결과 저장 완료


In [ ]:
# ── 전체 토론 발언 출력 ────────────────────────────────────────────────────
history = debate_pipeline.get_history(debate_id)
print_debate_history(history, title=f"AI vs AI 토론  |  {POLICY_CARD['title']}")



════════════════════════════════════════════════════════════════
  AI vs AI 토론  |  청년 월세 보조금 확대 법안  (20개 발언)
════════════════════════════════════════════════════════════════

[01] ✅ 찬성  🤖 찬성 AI  ·  주장
────────────────────────────────────────────────────────────────
  청년 월세 보조금 확대 법안은 청년들의 주거 부담을 경감시키고 경제적 자립을 돕는 중요한 정책입니다. 월 20만원의 지원은 소득 기준 중위
  150% 이하의 청년들에게 큰 실질적 혜택을 제공하여 사회적 안정성을 높이고, 주거 비용으로 인한 청년들의 생활 압박을 완화시킬 것입니다.
  [출처: 국민연금법 일부개정법률안]

[02] ❌ 반대  🤖 반대 AI  ·  주장
────────────────────────────────────────────────────────────────
  청년 월세 보조금 확대 법안은 재정적 부담이 크고 지속 가능성이 낮습니다. 지원 대상이 소득 기준 중위 150% 이하로 한정되어 모든 청년에게
  혜택을 제공하지 못하며, 자원의 비효율적 분배로 이어질 수 있습니다. 효과적인 장기적 해법을 위해 주택 공급 확대와 청년 고용 촉진 등 근본적
  문제가 함께 해결되어야 합니다. [출처: 청년고용촉진 특별법 일부개정법률안]

[03] ✅ 찬성  🤖 찬성 AI  ·  주장
────────────────────────────────────────────────────────────────
  청년 월세 보조금 확대 법안은 재정적 부담보다 청년들에게 주는 장기적 혜택이 중요합니다. 월 20만원 지원은 일정 수준의 경제적 완충 역할을 할
  수 있으며, 이는 청년들이 더 나은 주거 환경에서 생활할 수 있게 하여 사회적 안정감을 제공합니다. 또한, 청년 주거 불안은 청년 

: 

In [ ]:
# ── 주장 다지기 출력 ────────────────────────────────────────────────────────
print_summary(result.get("summary", {}), mode="AI vs AI")



════════════════════════════════════════════════════════════════
  ━━ 주장 다지기 (AI vs AI) ━━
════════════════════════════════════════════════════════════════

[전체 요약]
  토론에서는 청년 월세 보조금 확대 법안의 단기적 경제적 안정 지원과 장기적 경제 활성화 기여 가능성을 강조하는 찬성 측과, 보조금의
  근본적 문제 해결 한계와 시장 왜곡 가능성을 제기하는 반대 측이 대립했다. 찬성 측은 청년들의 주거비 부담 경감을 주장한 반면, 반대
  측은 주거 및 경제적 문제의 지속 가능한 해결책 필요성을 지적하였다.

[찬성 측 핵심 주장]
  • 청년 월세 보조금 확대는 경제적 안정을 도모하며, 청년들이 자립할 수 있는 환경을 조성한다.
  • 보조금을 통해 청년들의 주거비 부담을 줄이고, 더 나은 일자리 탐색과 직업 훈련에 집중할 수 있다.

[반대 측 핵심 주장]
  • 청년 월세 보조금 확대는 단기적인 안도감만 제공하며, 근본적인 주거 및 경제적 문제 해결책이 아니다.
  • 보조금 정책은 주거 시장의 왜곡을 초래하고 임대료 상승을 부추길 수 있다.

════════════════════════════════════════════════════════════════



## 3. AI vs User 토론

In [ ]:
TEST_USER_ID = 1

# ── 입장 / 난이도 선택 ────────────────────────────────────────────────────
print("=" * 40)
print("  AI vs User 토론 설정")
print("=" * 40)

_stance = input("나의 입장을 선택하세요  [pro(찬성) / con(반대)]  기본값=pro: ").strip().lower()
user_stance = _stance if _stance in ("pro", "con") else "pro"

_diff = input("난이도를 선택하세요  [easy / hard]  기본값=hard: ").strip().lower()
difficulty = _diff if _diff in ("easy", "hard") else "hard"

debate_id_u, user_p, ai_p = debate_pipeline.create_ai_vs_user(
    user_id=TEST_USER_ID,
    selected_policy=POLICY_CARD["title"],
    user_stance=user_stance,
    difficulty=difficulty,
)

print(f"\nAI vs User 토론방: #{debate_id_u}")
print(f"  사용자 입장: {user_stance.upper()}  |  AI 입장: {'con' if user_stance=='pro' else 'pro'}")
print(f"  난이도: {difficulty}")


  AI vs User 토론 설정

AI vs User 토론방: #11
  사용자 입장: PRO  |  AI 입장: con
  난이도: easy


In [ ]:
# ── AI 입장 발표 ─────────────────────────────────────────────────────────
ai_position = debate_pipeline.generate_ai_position(
    debate_id_u, POLICY_CARD, ai_p, difficulty=difficulty
)

print("\n" + "═" * 64)
print(f"  🤖 AI 입장 발표  ({'반대' if user_stance=='pro' else '찬성'})")
print("═" * 64)
for line in __import__('textwrap').wrap(ai_position, width=80):
    print(f"  {line}")
print()



════════════════════════════════════════════════════════════════
  🤖 AI 입장 발표  (반대)
════════════════════════════════════════════════════════════════
  청년 월세 보조금 확대 법안은 청년들의 주거비 부담을 실질적으로 경감시켜 경제적 안정을 도모하는 중요한 정책입니다. 소득 기준 중위 150%
  이하의 청년에게 월 20만원 보조금을 지원함으로써, 생활 안정을 돕고 경제 자립을 지원하는 긍정적인 효과를 기대할 수 있습니다.



In [ ]:
# ── 사용자 입력 라운드 ────────────────────────────────────────────────────
# 턴마다 메시지 유형이 자동으로 결정됩니다 (홀수=주장, 짝수=반박)
# 아무것도 입력하지 않고 Enter를 누르면 토론을 종료합니다.

MAX_TURNS = 6   # 최대 진행 턴 수 (변경 가능)
MSG_TYPES = ["argument", "rebuttal", "argument", "rebuttal", "argument", "rebuttal"]
TYPE_KR   = {"argument": "주장", "rebuttal": "반박"}

print("\n" + "═" * 64)
print(f"  📝 토론 시작  —  주제: {POLICY_CARD['title']}")
print(f"  나의 입장: {'찬성 ✅' if user_stance=='pro' else '반대 ❌'}  |  난이도: {difficulty}")
print("  (빈 입력 후 Enter → 토론 종료)")
print("═" * 64)

for turn_num in range(1, MAX_TURNS + 1):
    msg_type = MSG_TYPES[turn_num - 1]
    print(f"\n── 턴 {turn_num}  [{TYPE_KR[msg_type]}] ──────────────────────────────")
    user_msg = input(f"👤 내 발언을 입력하세요: ").strip()

    if not user_msg:
        print("  (입력 없음 — 토론을 종료합니다)")
        break

    print(f"👤 나: {user_msg}")
    print("🤖 AI 응답 생성 중...")

    ai_reply = debate_pipeline.process_user_message(
        debate_id=debate_id_u,
        user_participant_id=user_p,
        ai_participant_id=ai_p,
        user_message=user_msg,
        policy_card=POLICY_CARD,
        msg_type=msg_type,
        turn_num=turn_num,
        difficulty=difficulty,
    )

    print(f"\n🤖 AI:")
    for line in __import__('textwrap').wrap(ai_reply, width=80):
        print(f"  {line}")

else:
    print(f"\n  (최대 {MAX_TURNS}턴 완료)")

print()



════════════════════════════════════════════════════════════════
  📝 토론 시작  —  주제: 청년 월세 보조금 확대 법안
  나의 입장: 찬성 ✅  |  난이도: easy
  (빈 입력 후 Enter → 토론 종료)
════════════════════════════════════════════════════════════════

── 턴 1  [주장] ──────────────────────────────
👤 나: 저는 좋다고 생각합니다. 왜냐하면 제가 지원받고 싶거든요
🤖 AI 응답 생성 중...

🤖 AI:
  물론 개인적으로 지원을 받는 것은 좋은 일이지만, 제도의 효과는 개인의 필요를 넘어 전체 사회 및 경제적 영향을 고려해야 합니다. 청년 월세
  보조금 확대는 청년인의 재정적 부담을 줄이는 데 긍정적인 영향을 미칠 수 있을지 모르지만, 한정된 예산 내에서 이러한 지원이 장기적으로 지속
  가능한지 검토할 필요가 있습니다. 정부 예산을 효율적으로 사용하여 최대한 많은 청년에게 도움이 될 수 있는 방식인지 고려해야 합니다. 또한, 이
  보조금이 청년 고용 문제나 주거 안정과 같이 근본적인 문제를 해결하지 못할 가능성도 있습니다. 따라서, 단순히 금전적 지원을 넘어 더 포괄적인
  정책이 필요합니다.  [출처: (추가 확인 필요)]

── 턴 2  [반박] ──────────────────────────────
👤 나: q
🤖 AI 응답 생성 중...

🤖 AI:
  💬 사용자 반박은(는) 선택한 정책 주제와 관련된 내용이어야 합니다.

── 턴 3  [주장] ──────────────────────────────
👤 나: 청년 월세 보조금 확대는 단기적으로만 유효하다는 주장은 정책의 광범위한 영향을 과소평가하는 것입니다. 일시적인 재정 지원이지만, 이는 청년들이   주거비 걱정 없이 경제적 기회를 탐색할 시간을 제공합니다. 이는 장기적으로 경제 참여를 촉진하고, 결과적으로 사회 전체의 경제 

In [ ]:
# ── 주장 다지기 (사용자 피드백 포함) ────────────────────────────────────
summary_u = debate_pipeline.generate_summary(debate_id_u, POLICY_CARD)
print_summary(summary_u, mode="AI vs User")

Path("./output/debate_ai_vs_user.json").write_text(
    json.dumps(summary_u, ensure_ascii=False, indent=2)
)
print("결과 저장 완료  →  ./output/debate_ai_vs_user.json")


## 4. 전체 파이프라인 흐름 요약

```
01_setup_and_index.ipynb
  └─ JSONL → RDB(articles) → ChromaDB 6개 컬렉션 → 검색 품질 평가 → best_combo.txt

02_card_generation.ipynb
  └─ CardGenerationPipeline.run_daily()
     → articles/policies/bills 로드 → 필터 → 클러스터링
     → 카드 생성 → 편향 검사 → CARDS/CARD_TABS/BIAS_LOGS 저장 → ChromaDB upsert

03_chatbot_rag.ipynb
  └─ ChatbotRAGPipeline
     → 금칙어 필터 → 기사+카드 RAG 검색 → 히스토리
     → 응답 생성 → CHAT_SESSIONS/CHAT_MESSAGES 저장

04_debate.ipynb
  └─ DebatePipeline
     → AI vs AI: create_ai_vs_ai() → run_ai_vs_ai_full()
     → AI vs User: create_ai_vs_user() → process_user_message() × N → generate_summary()
     → DEBATES/DEBATE_PARTICIPANTS/DEBATE_MESSAGES 저장
```